# Entendimiento conjunto de los datos de pádel

Este notebook revisa simultáneamente los tres archivos de `Data/`: la base de videos, las respuestas de jugadores y la traza de partidos. Resume sus variables y calidad, explora una muestra del archivo grande y calcula el perfil completo de videos por bloques para no cargarlo entero en memoria.

## Lectura inicial de los archivos

- **Base de videos:** observaciones por jugador/frame con variables de tiempo, ubicaciones, pose, movimiento y controles de plausibilidad.
- **Respuestas de jugadores:** datos de formulario y algunas métricas agregadas de participación/rendimiento.
- **Traza de partidos:** puntos, marcador, composición de equipos y posiciones.

Se observaron formatos distintos: videos en UTF-8 con coma; formulario y traza en Windows-1252 con punto y coma. Las relaciones por nombre o por partido/punto son candidatas, no una llave confirmada. Las variables y unidades deben verificarse con quien generó los archivos.

In [ ]:
from pathlib import Path
from collections import Counter
import re
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 150)

DATA_DIR = Path.cwd() / 'Data'
ARCHIVOS = {
    'videos': {
        'ruta': DATA_DIR / 'Base_Videos_Final_202602.csv',
        'sep': ',', 'encoding': 'utf-8-sig',
    },
    'jugadores': {
        'ruta': DATA_DIR / 'Datos_Jugadores_Padel_202602(Respuestas de formulario 1) (1).csv',
        'sep': ';', 'encoding': 'cp1252',
    },
    'traza': {
        'ruta': DATA_DIR / 'Traza_Partidos_202602(CANCHA_2) (1).csv',
        'sep': ';', 'encoding': 'cp1252',
    },
}
CHUNK_SIZE = 100_000
FILAS_MUESTRA_VIDEOS = 10_000

for nombre, spec in ARCHIVOS.items():
    if not spec['ruta'].is_file():
        raise FileNotFoundError(f"No se encontró el archivo de {nombre}: {spec['ruta']}")

display(pd.DataFrame([
    {'archivo': nombre, 'ruta': str(spec['ruta']), 'tamaño_MB': round(spec['ruta'].stat().st_size / 1024**2, 2),
     'separador': repr(spec['sep']), 'codificación': spec['encoding']}
    for nombre, spec in ARCHIVOS.items()
]))

## 1. Lectura y limpieza estructural

Los archivos de formulario y traza son pequeños y se leen completos. En la traza se eliminan columnas sin nombre y filas completamente vacías, que pueden provenir del rango exportado desde una hoja de cálculo. La base de videos sólo se carga en una muestra en esta etapa.

In [ ]:
def leer_archivo_pequeno(nombre):
    spec = ARCHIVOS[nombre]
    df = pd.read_csv(
        spec['ruta'], sep=spec['sep'], encoding=spec['encoding'],
        low_memory=False, na_values=['#N/D'],
    )
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, (df.columns != '') & ~pd.Index(df.columns).str.startswith('Unnamed:')]
    return df.dropna(how='all').reset_index(drop=True)

jugadores = leer_archivo_pequeno('jugadores')
traza = leer_archivo_pequeno('traza')
muestra_videos = pd.read_csv(
    ARCHIVOS['videos']['ruta'], sep=',', encoding='utf-8-sig',
    nrows=FILAS_MUESTRA_VIDEOS, low_memory=False,
)

print(f"Filas de formulario: {len(jugadores):,}; columnas: {jugadores.shape[1]}")
print(f"Filas de traza con datos: {len(traza):,}; columnas útiles: {traza.shape[1]}")
print(f"Filas de muestra de videos: {len(muestra_videos):,}; columnas: {muestra_videos.shape[1]}")
display(traza.head(5))

## 2. Inventario de variables de los tres archivos

El diccionario reúne todas las columnas reales. La familia y descripción son hipótesis generadas a partir del nombre de la variable; no sustituyen un diccionario oficial del proceso de captura.

In [ ]:
def familia_video(col):
    n = col.lower()
    if any(x in n for x in ('frame', 'duration', 'time_since', 'schema')): return 'Video y tiempo'
    if any(x in n for x in ('player_id', 'player_name', 'team', 'nombre_norm')): return 'Jugador/equipo'
    if any(x in n for x in ('ball_position', 'player_position', 'distance_', 'zone')): return 'Cancha, posiciones y distancias'
    if any(x in n for x in ('speed', 'velocity', 'acceleration', 'displacement', 'prev_')): return 'Movimiento y velocidades'
    if any(x in n for x in ('implausible', 'available', 'confidence', 'hits_ball')): return 'Indicadores de calidad/evento'
    if any(x in n for x in ('head_', 'shoulder_', 'elbow_', 'wrist_', 'racket_', 'arm_', 'forearm_', 'body_scale', 'pose_')): return 'Pose, brazo y raqueta'
    if n in ('partido', 'punto'): return 'Partido y punto'
    return 'Otras variables de video'

def familia_formulario(col):
    n = col.lower()
    if any(x in n for x in ('nombre', 'codigo', 'marca temporal')): return 'Identificación/registro'
    if any(x in n for x in ('edad', 'genero', 'ciudad', 'programa', 'semestre')): return 'Perfil demográfico/académico'
    if any(x in n for x in ('experiencia', 'tiempo_jugando', 'nivel_actual', 'clases', 'interes')): return 'Experiencia e interés en pádel'
    if any(x in n for x in ('deporte', 'estado_fisico', 'frecuencia', 'estatura', 'talla', 'mano', 'ritmo', 'lesiones')): return 'Actividad física y salud declarada'
    if any(x in n for x in ('razon', 'objetivo', 'desayuno', 'musical', 'comentarios')): return 'Preferencias y respuestas abiertas'
    if any(x in n for x in ('puntos', 'partidos')): return 'Métricas agregadas de participación'
    return 'Otras respuestas de formulario'

def familia_traza(col):
    n = col.lower()
    if n.startswith('id_'): return 'Identificadores de partido/punto/jugador'
    if 'punto_equipo' in n or 'marcador' in n: return 'Punto y marcador'
    if 'banda' in n or 'equipo' in n: return 'Equipo y color/banda'
    if 'posicion' in n: return 'Posición en cancha'
    return 'Otras variables de la traza'

def descripcion_inicial(col):
    n = col.lower()
    if n.endswith('_x') or n.endswith('_y'): return 'Componente de coordenada; sistema y unidad por confirmar.'
    if 'speed' in n or 'velocity' in n: return 'Medida de velocidad; unidad y método por confirmar.'
    if 'acceleration' in n: return 'Medida de aceleración; unidad y método por confirmar.'
    if 'angle' in n or 'orientation' in n or 'direction' in n: return 'Ángulo/orientación; referencia y unidad por confirmar.'
    if 'distance' in n or 'length' in n or 'displacement' in n: return 'Distancia/longitud/desplazamiento; unidad por confirmar.'
    return 'Revisar categorías, definición y reglas de captura.'

tablas = {
    'videos': muestra_videos,
    'jugadores': jugadores,
    'traza': traza,
}
familias = {'videos': familia_video, 'jugadores': familia_formulario, 'traza': familia_traza}
diccionario = pd.DataFrame([
    {'archivo': archivo, 'variable': col, 'familia inferida': familias[archivo](col),
     'tipo en muestra/archivo': str(df[col].dtype),
     'descripción inicial': descripcion_inicial(col)}
    for archivo, df in tablas.items() for col in df.columns
])
print(f"Variables inventariadas entre los tres archivos: {len(diccionario)}")
display(diccionario)

## 3. Tipos, faltantes y resúmenes descriptivos

Los resúmenes de videos corresponden a la muestra de hasta 10.000 filas; los de formulario y traza corresponden a sus filas no vacías. Se omiten identificadores personales y respuestas abiertas al mostrar frecuencias.

In [ ]:
for nombre, df in tablas.items():
    print(f'\n### {nombre.upper()} — resumen de variables')
    resumen = pd.DataFrame({
        'tipo': df.dtypes.astype(str),
        'no nulos': df.notna().sum(),
        'faltantes': df.isna().sum(),
        'faltantes_%': (df.isna().mean() * 100).round(2),
        'valores_distintos': df.nunique(dropna=True),
    }).sort_values('faltantes_%', ascending=False)
    display(resumen)
    display(df.describe(include='all').T)

for nombre, df in [('jugadores', jugadores), ('traza', traza), ('videos (muestra)', muestra_videos)]:
    print(f'\n### Categorías seleccionadas: {nombre}')
    candidatas = [c for c in df.columns if c.lower() in {
        'genero', 'ciudad', 'programa_academico', 'experiencia_padel', 'nivel_actual_padel',
        'estado_fisico', 'mano_dominante', 'team', 'zone', 'racket_direction',
        'equipo1_banda1', 'equipo2_banda1', 'posicion_jugador_1_equipo_1', 'partido', 'punto'
    }]
    for col in candidatas:
        print(f'{col}:')
        display(df[col].value_counts(dropna=False).head(12).rename('frecuencia').to_frame())

## 4. Perfil completo de videos en streaming

Esta pasada lee el CSV grande por bloques y calcula filas, faltantes, categorías principales, nombres normalizados y llaves candidatas de partido/punto. No conserva las filas de video en memoria. Puede tardar varios minutos.

In [ ]:
def normalizar_nombre(valor):
    if pd.isna(valor): return ''
    texto = unicodedata.normalize('NFKD', str(valor)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', texto).strip().casefold()

def normalizar_id(valor):
    if pd.isna(valor): return ''
    texto = str(valor).strip()
    try:
        numero = float(texto)
        if numero.is_integer(): return str(int(numero))
    except ValueError:
        pass
    return texto.casefold()

columnas_video = list(muestra_videos.columns)
faltantes_video = pd.Series(0, index=columnas_video, dtype='int64')
categoricas_video = [c for c in ['player_name', 'team', 'zone', 'racket_direction', 'partido', 'punto', 'nombre_norm'] if c in columnas_video]
frecuencias_video = {c: Counter() for c in categoricas_video}
nombres_video = set()
llaves_video = set()
marcados_video = Counter()
filas_video = 0

for bloque in pd.read_csv(
    ARCHIVOS['videos']['ruta'], sep=',', encoding='utf-8-sig',
    chunksize=CHUNK_SIZE, low_memory=False,
):
    filas_video += len(bloque)
    faltantes_video += bloque.isna().sum().astype('int64')
    for c in categoricas_video:
        frecuencias_video[c].update(bloque[c].fillna('<FALTANTE>').astype(str).value_counts().to_dict())
    col_nombre = 'nombre_norm' if 'nombre_norm' in bloque.columns else 'player_name'
    nombres_video.update(x for x in bloque[col_nombre].map(normalizar_nombre).unique() if x)
    if {'partido', 'punto'}.issubset(bloque.columns):
        pares = bloque[['partido', 'punto']].dropna().drop_duplicates()
        llaves_video.update(
            (normalizar_id(a), normalizar_id(b)) for a, b in pares.itertuples(index=False, name=None)
        )
    for c in ['row_implausible', 'player_speed_implausible', 'ball_speed_implausible', 'pose_available']:
        if c in bloque.columns:
            marcados_video[c] += int(bloque[c].astype(str).str.lower().isin(['true', '1', 'yes', 'si']).sum())

print(f'Filas de video totales: {filas_video:,}')
display(pd.DataFrame({
    'faltantes': faltantes_video,
    'faltantes_%': (faltantes_video / filas_video * 100).round(2),
}).sort_values('faltantes_%', ascending=False))

for col, conteos in frecuencias_video.items():
    print(f'\n{col} — 15 categorías más frecuentes en el archivo completo')
    display(pd.Series(conteos).sort_values(ascending=False).head(15).rename('frecuencia').to_frame())

display(pd.DataFrame({
    'filas marcadas': pd.Series(marcados_video),
    'porcentaje': (pd.Series(marcados_video) / filas_video * 100).round(2),
}))

## 5. ¿Cómo se pueden relacionar las tres fuentes?

Se prueban coincidencias exactas de nombres después de quitar tildes, espacios exteriores y diferencias entre mayúsculas/minúsculas. También se compara `partido` + `punto` de videos con `ID_PARTIDO` + `ID_PUNTO` de la traza. Estas pruebas indican posibles enlaces; no prueban que los identificadores tengan la misma definición ni que el grano de las tablas sea idéntico.

In [ ]:
col_jugador_traza = [c for c in traza.columns if c.upper().startswith('ID_JUGADOR_')]
nombres_formulario = set(jugadores['NOMBRE'].dropna().map(normalizar_nombre)) if 'NOMBRE' in jugadores else set()
nombres_traza = set()
for col in col_jugador_traza:
    nombres_traza.update(x for x in traza[col].dropna().map(normalizar_nombre) if x)

def fila_llave(row, col_a, col_b):
    a, b = normalizar_id(row[col_a]), normalizar_id(row[col_b])
    return (a, b) if a and b else None

llaves_traza = set()
if {'ID_PARTIDO', 'ID_PUNTO'}.issubset(traza.columns):
    for _, fila in traza.iterrows():
        llave = fila_llave(fila, 'ID_PARTIDO', 'ID_PUNTO')
        if llave: llaves_traza.add(llave)

coincidencias_nombre = {
    'formulario vs traza': nombres_formulario & nombres_traza,
    'videos vs formulario': nombres_video & nombres_formulario,
    'videos vs traza': nombres_video & nombres_traza,
}
display(pd.DataFrame([
    {'comparación': nombre, 'distintos en A': len(a), 'distintos en B': len(b),
     'coincidencias exactas normalizadas': len(comunes)}
    for (nombre, comunes), (a, b) in zip(
        coincidencias_nombre.items(),
        [(nombres_formulario, nombres_traza), (nombres_video, nombres_formulario), (nombres_video, nombres_traza)],
    )
]))

coincidencias_llave = llaves_video & llaves_traza
print(f'Llaves distintas de partido-punto en videos: {len(llaves_video):,}')
print(f'Llaves distintas de partido-punto en traza: {len(llaves_traza):,}')
print(f'Llaves coincidentes: {len(coincidencias_llave):,}')
print('No se hace un merge automático: primero confirma que los nombres y los identificadores se refieren a las mismas personas/partidos.')

## 6. Revisión de calidad y distribuciones de muestra

La encuesta y la traza se inspeccionan completas; los duplicados de video se revisan sólo en la muestra, por el tamaño del CSV. Los histogramas también describen la muestra, no la base completa.

In [ ]:
print(f'Filas duplicadas exactas en formulario: {jugadores.duplicated().sum():,}')
print(f'Filas duplicadas exactas en traza ya limpia: {traza.duplicated().sum():,}')
print(f'Filas duplicadas exactas en muestra de videos: {muestra_videos.duplicated().sum():,} de {len(muestra_videos):,}')

numericas_jugadores = jugadores.select_dtypes(include='number').columns.tolist()
if numericas_jugadores:
    display(jugadores[numericas_jugadores].describe(percentiles=[.01, .25, .5, .75, .99]).T)
numericas_videos = [c for c in ['player_speed_mps', 'ball_speed_mps', 'pose_confidence', 'distance_player_to_ball_m']
                    if c in muestra_videos and pd.api.types.is_numeric_dtype(muestra_videos[c])]
if numericas_videos:
    muestra_videos[numericas_videos].hist(bins=30, figsize=(12, 8), layout=(-1, 2))
    plt.suptitle('Distribuciones seleccionadas de videos (muestra)')
    plt.tight_layout()
    plt.show()

if 'NIVEL_ACTUAL_PADEL' in jugadores.columns:
    jugadores['NIVEL_ACTUAL_PADEL'].value_counts(dropna=False).plot(kind='bar', figsize=(8, 4), title='Nivel actual declarado (formulario)')
    plt.ylabel('Respuestas')
    plt.tight_layout()
    plt.show()
if 'MARCADOR' in traza.columns:
    traza['MARCADOR'].value_counts().head(15).plot(kind='bar', figsize=(10, 4), title='Marcadores más frecuentes (traza)')
    plt.ylabel('Puntos/filas')
    plt.tight_layout()
    plt.show()

## 7. Conclusiones iniciales y preguntas antes de modelar

1. Las fuentes parecen tener granos diferentes: observación por frame/jugador, respuesta por participante y evento/punto del partido.
2. Los archivos pequeños usan `#N/D` y pueden contener filas/columnas vacías de exportación; se deben distinguir de valores válidos antes de analizar.
3. Los nombres no son una clave segura: puede haber nombres abreviados, apellidos omitidos o variantes de escritura. El conteo de coincidencias de arriba es sólo coincidencia textual exacta tras normalizar.
4. Antes de unir, confirmar qué representa cada fila, cómo se construyen `partido` y `punto`, la zona horaria del formulario y las unidades/sistemas de coordenadas de la base de videos.
5. Definir reglas de tratamiento para métricas agregadas con `#N/D`, faltantes de salud, estados de pose y filas marcadas como implausibles.

El siguiente paso recomendado es obtener un identificador común y documentado de participante y una correspondencia oficial entre partido/punto de la traza y los videos.